In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip#, DeltaTable
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, to_timestamp
from helpers import get_table, get_bronze
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from functools import reduce
print(pyspark.__version__)

3.5.8


In [2]:
builder = (
    SparkSession.builder
    .appName("silver_processing")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(
    builder,
    extra_packages=["org.postgresql:postgresql:42.7.3"]
).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

26/04/27 19:05:32 WARN Utils: Your hostname, CVPThuyTTD11-1 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/27 19:05:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thuyttd11/.local/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thuyttd11/.ivy2/cache
The jars for the packages stored in: /home/thuyttd11/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.postgresql#postgresql added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c93b2702-ea09-4573-a2fb-10eda36d3812;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in central
	found io.delta#delta-storage;3.3.2 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.postgresql#postgresql;42.7.3 in central
	found org.checkerframework#checker-qual;3.42.0 in central
:: resolution report :: resolve 618ms :: artifacts dl 34ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.3.2 from central in [default]
	io.delta#delta-storage;3.3.2 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.checkerframework#checker-qual;3.42.0 from central in [default]
	org.postgresql#postgresql;42.7.3 from central in [default]
	------------------------

## Users silver
+ Define grain: one row per user_id => remove duplicates, validate uniqueness for user_id
+ Enforce schema/types for id, user_id, first_name, last_name, email, country, age, gender, acquisition_channel, is_enterprise, created_at, ingest_time, source_identifier, batch_id
+ Trim whitespace for user_id, first_name, last_name, email, country, gender, acquisition_channel
+ Standardize text case for email, country, gender, acquisition_channel
+ Validate required/non-null fields for user_id, created_at
+ Normalize and validate email
+ Standardize country to predefined categories plus Other
+ Validate age within range 18–100
+ Normalize gender to allowed values
+ Normalize acquisition_channel to {organic, paid, referral}
+ Normalize and validate boolean values for is_enterprise
+ Validate lineage/control fields for source_identifier, batch_id
+ Remove or quarantine invalid/out-of-domain records
+ Ensure join-key compatibility on user_id with support_tickets, usage_events, subscriptions
+ Derive created_year, created_month from created_at
+ Derive user_tenure_days, user_tenure_months from created_at
+ Optionally derive age_group from age
+ Decide whether to keep or drop id in final silver_users
+ Keep lineage metadata columns ingest_time, source_identifier, batch_id if needed
+ Write final output as Delta table
+ Define load strategy: full refresh or incremental/upsert on user_id
+ Record data quality metrics for duplicates, null user_id, invalid email, invalid age, invalid country, invalid gender, invalid acquisition_channel, invalid is_enterprise, rejected rows, output rows

In [3]:
users_bronze = "/home/thuyttd11/subscription-analytics/spark-data/bronze/users"
df_bronze_users = get_bronze(users_bronze, spark=spark).drop("source_identifier", "id", "batch_id")
df_bronze_users.show(2, truncate=False)

[Stage 8:>                                                          (0 + 1) / 1]

+-------+----------+---------+------------------------+-------+---+------+-------------------+-------------+-------------------+--------------------------+
|user_id|first_name|last_name|email                   |country|age|gender|acquisition_channel|is_enterprise|created_at         |ingest_time               |
+-------+----------+---------+------------------------+-------+---+------+-------------------+-------------+-------------------+--------------------------+
|1      |Anthony   |Patil    |anthony.patil@gmail.com |India  |37 |Male  |referral           |false        |2024-06-09 04:21:04|2026-04-22 11:25:46.763036|
|2      |Linda     |Miller   |linda.miller@outlook.com|US     |29 |Male  |referral           |true         |2023-10-31 07:47:20|2026-04-22 11:25:46.763036|
+-------+----------+---------+------------------------+-------+---+------+-------------------+-------------+-------------------+--------------------------+
only showing top 2 rows



### Remove duplicates based on user_id

In [4]:
# Remove duplicates based on user_id, created_at and latest ingestion_time

# Business grain: user_id + created_at
window_spec = (
    Window
    .partitionBy("user_id", "created_at")
    .orderBy(F.col("ingest_time").desc())
)

df_with_rn = df_bronze_users.withColumn(
    "rn", F.row_number().over(window_spec)
)

# Valid (deduplicated)
df1 = (
    df_with_rn
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Quarantine (duplicates)
df1_quarantine = (
    df_with_rn
    .filter(F.col("rn") > 1)
    .drop("rn")
)

# Checks
print("Valid count:", df1.count())
print("Quarantine count:", df1_quarantine.count())

[Stage 11:>                                                         (0 + 1) / 1]

Valid count: 50000


Quarantine count: 50000


### Enforce schema

In [5]:
df1 = (
    df1
    .withColumn("user_id", F.col("user_id").cast("int"))
    .withColumn("first_name", F.col("first_name").cast("string"))
    .withColumn("last_name", F.col("last_name").cast("string"))
    .withColumn("email", F.col("email").cast("string"))
    .withColumn("country", F.col("country").cast("string"))
    .withColumn("age", F.col("age").cast("int"))
    .withColumn("gender", F.col("gender").cast("string"))
    .withColumn("acquisition_channel", F.col("acquisition_channel").cast("string"))
    .withColumn("is_enterprise", F.col("is_enterprise").cast("boolean"))
    .withColumn("created_at", F.to_timestamp("created_at"))
    .withColumn("ingest_time", F.to_timestamp("ingest_time"))
)


### Trim whitespace for text columns

In [6]:
# Not care about NULL, Spark will keep it that way
df2 = (
    df1
    .withColumn("first_name", F.trim(F.col("first_name")))
    .withColumn("last_name", F.trim(F.col("last_name")))
    .withColumn("email", F.trim(F.col("email")))
    .withColumn("gender", F.trim(F.col("gender")))
    .withColumn("acquisition_channel", F.trim(F.col("acquisition_channel")))
)
df2.show(2)

+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
|user_id|first_name|last_name|               email|country|age|gender|acquisition_channel|is_enterprise|         created_at|         ingest_time|
+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
|      1|   Anthony|    Patil|anthony.patil@gma...|  India| 37|  Male|           referral|        false|2024-06-09 04:21:04|2026-04-22 11:25:...|
|      2|     Linda|   Miller|linda.miller@outl...|     US| 29|  Male|           referral|         true|2023-10-31 07:47:20|2026-04-22 11:25:...|
+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
only showing top 2 rows



### Standardize text case for email, country, gender, acquisition_channel

In [7]:
df3 = (
    df2
    .withColumn("email", F.lower(F.trim(F.col("email"))))
    .withColumn("country", F.initcap(F.trim(F.col("country"))))
    .withColumn("gender", F.lower(F.trim(F.col("gender"))))
    .withColumn("acquisition_channel", F.lower(F.trim(F.col("acquisition_channel"))))
)
df3.show(2)

+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
|user_id|first_name|last_name|               email|country|age|gender|acquisition_channel|is_enterprise|         created_at|         ingest_time|
+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
|      1|   Anthony|    Patil|anthony.patil@gma...|  India| 37|  male|           referral|        false|2024-06-09 04:21:04|2026-04-22 11:25:...|
|      2|     Linda|   Miller|linda.miller@outl...|     Us| 29|  male|           referral|         true|2023-10-31 07:47:20|2026-04-22 11:25:...|
+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+
only showing top 2 rows



### Validate required/non-null fields for user_id, created_at and quarantine invalid records for further analysis

In [8]:
df4 = df3.filter(
    F.col("user_id").isNotNull() & F.col("created_at").isNotNull()
)
df4.count()

50000

In [9]:
df4_quarantine = df3.filter(
    F.col("user_id").isNull() | F.col("created_at").isNull()
)

df4_quarantine.count()

0

### Standardize country to predefined categories plus Other

In [10]:
n_countries = df4.select("country").distinct().count()
df4.select("country").distinct().show(n_countries, truncate=False)

+---------+
|country  |
+---------+
|Singapore|
|Germany  |
|France   |
|India    |
|Other    |
|Spain    |
|Us       |
|Uk       |
|Canada   |
|Brazil   |
|Japan    |
|Australia|
|Vietnam  |
+---------+



In [11]:
predefined_countries = [
    "Australia",
    "Brazil",
    "Canada",
    "China",
    "Denmark",
    "France",
    "Germany",
    "India",
    "Indonesia",
    "Italy",
    "Japan",
    "Netherlands",
    "Norway",
    "Philippines",
    "Singapore",
    "South Korea",
    "Spain",
    "Sweden",
    "Thailand",
    "United Kingdom",
    "United States",
    "Vietnam"
]

df5 = (
    df4
    .withColumn("country_clean", F.initcap(F.trim(F.col("country"))))
    .withColumn(
        "is_valid_country",
        F.col("country_clean").isin(predefined_countries)
    )
    .withColumn(
        "country",
        F.when(F.col("is_valid_country"), F.col("country_clean"))
         .otherwise(F.lit("Other"))
    )
    .drop("country_clean")
)
# Checking
n_countries = df5.select("country").distinct().count()
df5.select("country").distinct().show(n_countries, truncate=False)

+---------+
|country  |
+---------+
|Singapore|
|Germany  |
|France   |
|India    |
|Other    |
|Spain    |
|Canada   |
|Brazil   |
|Japan    |
|Australia|
|Vietnam  |
+---------+



### Validate age within range 18–100, if not, remove

Business rule (Silver):

+ Age must be numeric
+ Age must be between 18 and 100
+ Invalid values → quarantine
+ Never silently coerce bad values

In [12]:
df6 = df5.filter(F.col("age").between(18,100))
df6_quarantine = df5.filter(~F.col("age").between(18,100)\
                                                    | F.col("age").isNull())

In [13]:
df6.count(), df6_quarantine.count()

(50000, 0)

### Normalize gender to allowed values
Business rules:

+ Gender must be in: female, male or other.
+ Quarantine null if any

In [14]:
df6.select("gender").distinct().show()

+------+
|gender|
+------+
|female|
| other|
|  male|
+------+



In [15]:
allowed_genders = ["female", "male", "other"]
df7 = df6.filter(F.col("gender").isin(allowed_genders))
df7_quarantine = df6.filter(~F.col("gender").isin(allowed_genders)\
                                                        | F.col("gender").isNull())
df7.select("gender").distinct().show(), \
df7_quarantine.select("gender").distinct().show()

+------+
|gender|
+------+
|female|
| other|
|  male|
+------+

+------+
|gender|
+------+
+------+



(None, None)

### Normalize acquisition_channel to {organic, paid, referral}

In [16]:
allowed_channels = ["organic", "paid", "referral"]
# valid
df8 = df7.filter(
    F.col("acquisition_channel").isin(allowed_channels)
)
# quarantine
df8_quarantine = df7.filter(
    ~F.col("acquisition_channel").isin(allowed_channels))
 
df8.select("acquisition_channel").distinct().show(), \
df8_quarantine.select("acquisition_channel").distinct().show()

+-------------------+
|acquisition_channel|
+-------------------+
|           referral|
|            organic|
|               paid|
+-------------------+

+-------------------+
|acquisition_channel|
+-------------------+
+-------------------+



(None, None)

### Normalize and validate boolean values for is_enterprise


In [17]:
df9 = df8.filter(F.col("is_enterprise").isNotNull())

df9_quarantine = df8.filter(F.col("is_enterprise").isNull())
df9.select("is_enterprise").distinct().show(), \
df9_quarantine.select("is_enterprise").distinct().show()

+-------------+
|is_enterprise|
+-------------+
|         true|
|        false|
+-------------+

+-------------+
|is_enterprise|
+-------------+
+-------------+



(None, None)

## Get all quarantine tables

In [18]:
quarantine_dfs = [
    df1_quarantine,
    df4_quarantine,
    df6_quarantine,
    df7_quarantine,
    df8_quarantine,
    df9_quarantine
    
]

df_quarantine_all = reduce(
    lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True),
    quarantine_dfs
)
df_quarantine_all.count()

50000

### Upsert strategy

Business rule:
The users Silver table should be loaded using an upsert strategy on user_id because user profile attributes can change over time (for example email, country, age, or enterprise status). The business usually needs one current, trusted record per user, not multiple duplicated versions of the same user across batches. Therefore, each new load should update existing users when user_id already exists and insert new users when user_id is new.

In [19]:
df_user_silver = df9

In [20]:
# Delta output path
silver_path = "./silver/users"

# Only keep the columns you want in the Silver Delta table
silver_cols = df_user_silver.columns

df_upsert = df_user_silver.select(*silver_cols)

# Optional but recommended:
# keep only one row per user_id in the incoming batch
# here: keep the latest record by created_at
w = Window.partitionBy("user_id").orderBy(F.col("created_at").desc(), F.col("ingest_time").desc())

df_upsert = (
    df_upsert
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# If Delta table already exists -> MERGE (upsert)
if DeltaTable.isDeltaTable(spark, silver_path):
    print("Delta table already exists")
    delta_target = DeltaTable.forPath(spark, silver_path)

    (
        delta_target.alias("target")
        .merge(
            df_upsert.alias("source"),
            "target.user_id = source.user_id"
        )
        # Update only if incoming row is newer
        .whenMatchedUpdate(
            condition="source.ingest_time > target.ingest_time",
            set={c: f"source.{c}" for c in silver_cols}
        )
        .whenNotMatchedInsert(
            values={c: f"source.{c}" for c in silver_cols}
        )
        .execute()
    )

# If Delta table does not exist yet -> initial save
else:
    print("Delta table does not exist yet")
    (
        df_upsert
        .write
        .format("delta")
        .mode("overwrite")
        .save(silver_path)
    )

Delta table does not exist yet


In [21]:
df_user_silver_read = (
    spark.read
    .format("delta")
    .load(silver_path)
)

df_user_silver_read.show(5)

[Stage 199:============================================>          (40 + 4) / 50]

+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+----------------+
|user_id|first_name|last_name|               email|country|age|gender|acquisition_channel|is_enterprise|         created_at|         ingest_time|is_valid_country|
+-------+----------+---------+--------------------+-------+---+------+-------------------+-------------+-------------------+--------------------+----------------+
|      1|   Anthony|    Patil|anthony.patil@gma...|  India| 37|  male|           referral|        false|2024-06-09 04:21:04|2026-04-22 11:25:...|            true|
|      6|      Elma|Bohlander|elma.bohlander@we...|Germany| 32|female|            organic|        false|2026-03-10 04:35:50|2026-04-22 11:25:...|            true|
|     12|     Donna|    Gomez|donna.gomez@gmail...|  Other| 24|  male|            organic|        false|2024-03-15 12:31:27|2026-04-22 11:25:...|           false|
|     13|   Remigio|  

In [22]:
spark.stop()